<div style="
    background: linear-gradient(135deg, #0d1117 0%, #161b22 60%, #0d1117 100%);
    border: 1px solid #21262d;
    border-left: 4px solid #00b4d8;
    border-radius: 10px;
    padding: 32px 36px 28px;
    font-family: 'JetBrains Mono', 'Fira Code', monospace;
    margin-bottom: 8px;
">
<div style="color:#00b4d8; font-size:11px; letter-spacing:3px; text-transform:uppercase; margin-bottom:10px;">
  Data Engineering · Web Scraping
</div>
<div style="color:#e6edf3; font-size:26px; font-weight:700; letter-spacing:1px; margin-bottom:6px;">
  YouTube Channel Scraper
</div>
<div style="color:#8b949e; font-size:14px; margin-bottom:20px;">
  Automated metadata extraction via Selenium — headless Chrome, infinite scroll, structured CSV export
</div>
<hr style="border:none; border-top:1px solid #21262d; margin:16px 0;">
<table style="width:100%; font-size:12px; color:#8b949e; border:none;">
  <tr>
    <td style="padding:4px 0;"><span style="color:#00b4d8;">Stack</span></td>
    <td style="padding:4px 16px;">Python 3 · Selenium 4 · webdriver-manager · Pandas</td>
    <td style="padding:4px 0;"><span style="color:#00b4d8;">Output</span></td>
    <td style="padding:4px 16px;"><code style="background:#21262d; padding:2px 6px; border-radius:4px; color:#e6edf3;">output/&lt;channel&gt;_videos_&lt;timestamp&gt;.csv</code></td>
  </tr>
  <tr>
    <td style="padding:4px 0;"><span style="color:#00b4d8;">Target</span></td>
    <td style="padding:4px 16px;"><code style="background:#21262d; padding:2px 6px; border-radius:4px; color:#e6edf3;">youtube.com/@AndroidBasha</code></td>
    <td style="padding:4px 0;"><span style="color:#00b4d8;">Author</span></td>
    <td style="padding:4px 16px;">Ali Essam — ali-essam2002</td>
  </tr>
</table>
</div>

## Pipeline Overview

```
┌─────────────────────────────────────────────────────────────────────┐
│  1. Install  →  2. Imports  →  3. Config  →  4. Driver              │
│     └─> 5. Helpers  →  6. Scraper Class  →  7. Execute              │
│           └─> 8. DataFrame  →  9. Quality  →  10. Explore           │
│                 └─> 11. Export CSV  →  12. Verify                   │
└─────────────────────────────────────────────────────────────────────┘
```

| # | Section | Description |
|---|---------|-------------|
| 1 | **Dependencies** | Install Selenium, webdriver-manager, Pandas |
| 2 | **Imports & Logging** | All modules + structured logger |
| 3 | **Configuration** | Channel URL, scroll params, output paths |
| 4 | **Chrome Driver Factory** | Headless Chrome with anti-detection flags |
| 5 | **Helper Utilities** | Safe element access, view parser, scroll engine |
| 6 | **YouTubeChannelScraper** | OOP scraper class with context manager |
| 7 | **Execution** | Run the scraper |
| 8 | **DataFrame Build** | Type casting, cleaning, deduplication |
| 9 | **Data Quality** | dtypes, null counts, view statistics |
| 10 | **Exploration** | Top videos, upload distribution |
| 11 | **CSV Export** | UTF-8-sig for Arabic-safe Excel output |
| 12 | **Verification** | Reload and sanity-check the saved file |

---
## § 1 · Install Dependencies

In [1]:
# Run once — installs all required packages
# Uncomment the line below if running in a fresh environment

! pip install selenium webdriver-manager pandas --quiet
! python -m pip install --upgrade pip


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 3.9 MB/s eta 0:00:01
   ---------------------------- ----------- 1.3/1.8 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 3.9 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.2.1
    Uninstalling pip-26.2.1:
      Successfully uninstalled pip-26.2.1


---
## § 2 · Imports & Logging

In [2]:
import time
import re
import logging
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Optional

# ── Selenium ──────────────────────────────────────────────────────────────────
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    StaleElementReferenceException,
)
from webdriver_manager.chrome import ChromeDriverManager

# ── Data ──────────────────────────────────────────────────────────────────────
import pandas as pd

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("yt_scraper")

logger.info("All imports loaded successfully.")

12:42:44  INFO      All imports loaded successfully.


---
## § 3 · Configuration

> Edit only this cell to point the scraper at a different channel.

In [3]:
# ── Target ────────────────────────────────────────────────────────────────────
CHANNEL_URL    = "https://www.youtube.com/@AndroidBasha/videos"
CHANNEL_HANDLE = "@AndroidBasha"

# ── Scraping Parameters ────────────────────────────────────────────────────────
MAX_VIDEOS   = 0     # 0 = scrape ALL videos; set a positive int to cap the run
SCROLL_PAUSE = 2.5   # seconds between each scroll step
PAGE_TIMEOUT = 30    # WebDriverWait timeout in seconds
MAX_RETRIES  = 3     # stale-element retry attempts per card

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
CSV_PATH  = OUTPUT_DIR / f"{CHANNEL_HANDLE.lstrip('@')}_videos_{timestamp}.csv"

logger.info(f"Target channel : {CHANNEL_URL}")
logger.info(f"Output path    : {CSV_PATH}")

12:42:44  INFO      Target channel : https://www.youtube.com/@AndroidBasha/videos
12:42:44  INFO      Output path    : output\AndroidBasha_videos_20261021_124244.csv


---
## § 4 · Chrome Driver Factory

Headless Chrome with anti-detection hardening. Pass `headless=False` to observe the browser live during development.

In [4]:
def build_driver(headless: bool = True) -> webdriver.Chrome:
    """
    Instantiate a Selenium Chrome WebDriver with anti-detection flags.

    Parameters
    ----------
    headless : bool
        Run Chrome without a visible window (default: True).

    Returns
    -------
    webdriver.Chrome
        A fully configured, ready-to-use driver instance.
    """
    opts = Options()

    if headless:
        opts.add_argument("--headless=new")

    chrome_flags = [
        "--no-sandbox",
        "--disable-dev-shm-usage",
        "--disable-gpu",
        "--window-size=1920,1080",
        "--lang=en-US,en;q=0.9",
        "--disable-blink-features=AutomationControlled",
        "--disable-extensions",
        "--disable-notifications",
        "--mute-audio",
    ]
    for flag in chrome_flags:
        opts.add_argument(flag)

    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    opts.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

    service = Service(ChromeDriverManager().install())
    driver  = webdriver.Chrome(service=service, options=opts)

    # Mask the `navigator.webdriver` flag in the JavaScript context
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"},
    )

    logger.info(f"Chrome driver ready  [headless={headless}]")
    return driver

---
## § 5 · Helper Utilities

| Function | Purpose |
|----------|---------|
| `safe_text` | Extract `.text` from a child element without raising |
| `safe_attr` | Extract an attribute value from a child element without raising |
| `parse_views` | Convert `'1.2M views'` / `'340K views'` → `int` |
| `scroll_to_bottom` | Scroll page until no new content; clicks continuation buttons |

In [5]:
def safe_text(parent, by: str, selector: str, default: str = "N/A") -> str:
    """Return `.text` of a child element, or *default* on any DOM exception."""
    try:
        return parent.find_element(by, selector).text.strip()
    except (NoSuchElementException, StaleElementReferenceException):
        return default


def safe_attr(parent, by: str, selector: str, attr: str, default: str = "N/A") -> str:
    """Return an attribute value of a child element, or *default* on any DOM exception."""
    try:
        return parent.find_element(by, selector).get_attribute(attr) or default
    except (NoSuchElementException, StaleElementReferenceException):
        return default


def parse_views(raw: str) -> Optional[int]:
    """
    Convert a raw view-count string to an integer.

    Examples
    --------
    >>> parse_views("1.2M views")   # → 1_200_000
    >>> parse_views("340K views")   # → 340_000
    >>> parse_views("N/A")          # → None
    """
    if raw == "N/A":
        return None
    raw = raw.lower().replace(",", "")
    multipliers = {"k": 1_000, "m": 1_000_000, "b": 1_000_000_000}
    match = re.search(r"([\.\d]+)\s*([kmb])?", raw)
    if not match:
        return None
    num, suffix = float(match.group(1)), match.group(2)
    return int(num * multipliers.get(suffix, 1))


def scroll_to_bottom(driver: webdriver.Chrome, max_scrolls: int = 999) -> None:
    """
    Scroll the page incrementally until no new content loads.

    Between each scroll, attempts to click any YouTube 'Show more' or
    continuation buttons to trigger deferred content rendering.

    Parameters
    ----------
    driver      : Selenium WebDriver instance
    max_scrolls : Upper bound on scroll attempts (safety guard)
    """
    last_height = driver.execute_script("return document.documentElement.scrollHeight")

    # CSS selectors for YouTube continuation / 'Show more' buttons
    continuation_selectors = [
        "button.yt-spec-button-shape-next[aria-label*='more']",
        "yt-next-continuation button",
        "ytd-continuation-item-renderer button",
        "paper-button#button[aria-label*='more']",
    ]

    for scroll_n in range(1, max_scrolls + 1):
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        time.sleep(SCROLL_PAUSE)

        for sel in continuation_selectors:
            try:
                btn = driver.find_element(By.CSS_SELECTOR, sel)
                driver.execute_script("arguments[0].click();", btn)
                logger.info(f"  Clicked continuation button: {sel}")
                time.sleep(SCROLL_PAUSE)
            except Exception:
                pass

        new_height = driver.execute_script("return document.documentElement.scrollHeight")
        if new_height == last_height:
            logger.info(f"  No new content after scroll #{scroll_n} — page fully loaded.")
            break

        logger.info(f"  Scroll {scroll_n:3d}  —  page height: {new_height:,}px")
        last_height = new_height


logger.info("Helper functions defined.")

12:42:44  INFO      Helper functions defined.


---
## § 6 · YouTubeChannelScraper

Object-oriented scraper that wraps the entire extraction workflow. Implements a context manager so the browser is always closed cleanly even if an exception occurs.

| Method | Responsibility |
|--------|---------------|
| `__enter__ / __exit__` | Context manager — auto-closes the browser |
| `_dismiss_cookies()` | Handles the GDPR / cookie consent popup |
| `_parse_card(card)` | Extracts all metadata fields from a single video card |
| `scrape(url, max_videos)` | Loads the page, triggers infinite scroll, collects all cards |

In [6]:
class YouTubeChannelScraper:
    """
    Scrape public video metadata from a YouTube channel's /videos tab.

    Usage
    -----
    with YouTubeChannelScraper(headless=True) as scraper:
        records = scraper.scrape(max_videos=0)  # 0 = all videos
    """

    # ── CSS selectors (verified via DevTools, June 2025) ──────────────────────
    _SEL_VIDEO_CARD = "ytd-rich-item-renderer"
    _SEL_TITLE_H3   = "h3.ytLockupMetadataViewModelHeadingReset"
    _SEL_URL_ANCHOR = "a.ytLockupViewModelContentImage"
    _SEL_META_ROW   = "div[role='group'].ytContentMetadataViewModelMetadataRow"
    _SEL_META_SPAN  = "span.ytAttributedStringHost"
    _SEL_DURATION   = (
        "yt-thumbnail-badge-view-model badge-shape, "
        "yt-thumbnail-badge-view-model badge-shape span, "
        "yt-thumbnail-bottom-overlay-view-model badge-shape, "
        "ytd-thumbnail-overlay-time-status-renderer span"          # legacy fallback
    )
    _SEL_COOKIE = 'button[aria-label*="Accept"]'

    # ─────────────────────────────────────────────────────────────────────────

    def __init__(self, headless: bool = True):
        self.driver = build_driver(headless)
        self.wait   = WebDriverWait(self.driver, PAGE_TIMEOUT)

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.driver.quit()
        logger.info("Browser closed.")
        return False  # do not suppress exceptions

    # ── Private helpers ───────────────────────────────────────────────────────

    def _dismiss_cookies(self) -> None:
        """Attempt to close the GDPR cookie consent popup (silent on miss)."""
        try:
            btn = WebDriverWait(self.driver, 8).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, self._SEL_COOKIE))
            )
            btn.click()
            logger.info("Cookie consent dismissed.")
        except TimeoutException:
            pass  # No popup present — continue normally

    def _parse_card(self, card) -> Dict:
        """
        Extract all available metadata from a single ytd-rich-item-renderer card.

        Returns
        -------
        dict with keys: title, url, views_raw, views, upload_date, duration,
                        channel, scraped_at
        """
        # ── Title ─────────────────────────────────────────────────────────────
        # Prefer the `title` attribute over `.text` — it avoids the duration suffix
        title = safe_attr(card, By.CSS_SELECTOR, self._SEL_TITLE_H3, "title", "N/A")
        if title == "N/A":
            title = safe_text(card, By.CSS_SELECTOR, self._SEL_TITLE_H3, "N/A")

        # ── URL ───────────────────────────────────────────────────────────────
        url_raw = safe_attr(card, By.CSS_SELECTOR, self._SEL_URL_ANCHOR, "href", "N/A")
        url = (
            f"https://www.youtube.com{url_raw}"
            if url_raw != "N/A" and url_raw.startswith("/")
            else url_raw
        )

        # ── Duration ──────────────────────────────────────────────────────────
        duration = "N/A"
        for sel in (s.strip() for s in self._SEL_DURATION.split(",") if s.strip()):
            try:
                el  = card.find_element(By.CSS_SELECTOR, sel)
                val = (el.get_attribute("aria-label") or el.text or "").strip()
                if val and val != "N/A":
                    duration = val
                    break
            except (NoSuchElementException, StaleElementReferenceException):
                continue

        # ── Views & Upload date ───────────────────────────────────────────────
        # Layout: div[role="group"]  →  span("2.9M views") · span("•") · span("2 weeks ago")
        views_raw  = "N/A"
        upload_raw = "N/A"
        try:
            for row in card.find_elements(By.CSS_SELECTOR, self._SEL_META_ROW):
                spans = [
                    s for s in row.find_elements(By.CSS_SELECTOR, self._SEL_META_SPAN)
                    if s.text.strip() and s.text.strip() != "•"
                ]
                if len(spans) >= 1:
                    views_raw = spans[0].text.strip()
                if len(spans) >= 2:
                    upload_raw = spans[1].text.strip()
                if views_raw != "N/A":
                    break
        except (NoSuchElementException, StaleElementReferenceException):
            pass

        return {
            "title":       title,
            "url":         url,
            "views_raw":   views_raw,
            "views":       parse_views(views_raw),
            "upload_date": upload_raw,
            "duration":    duration,
            "channel":     CHANNEL_HANDLE,
            "scraped_at":  datetime.utcnow().isoformat(),
        }

    # ── Public API ────────────────────────────────────────────────────────────

    def scrape(
        self,
        url: str = CHANNEL_URL,
        max_videos: int = MAX_VIDEOS,
    ) -> List[Dict]:
        """
        Navigate to *url*, scroll until all videos are loaded, then parse
        every video card into a list of dictionaries.

        Parameters
        ----------
        url        : Full URL of the channel's /videos page
        max_videos : Cap on number of videos to scrape (0 = unlimited)

        Returns
        -------
        list[dict]  — one dict per video card successfully parsed
        """
        logger.info(f"Navigating to: {url}")
        self.driver.get(url)
        self._dismiss_cookies()

        self.wait.until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, self._SEL_VIDEO_CARD))
        )
        logger.info("Page loaded — starting infinite scroll.")

        max_scrolls = 999 if max_videos == 0 else max(10, max_videos // 5)
        scroll_to_bottom(self.driver, max_scrolls)

        cards = self.driver.find_elements(By.CSS_SELECTOR, self._SEL_VIDEO_CARD)
        logger.info(f"Detected {len(cards)} video cards on page.")

        if max_videos:
            cards = cards[:max_videos]

        records: List[Dict] = []
        for i, card in enumerate(cards, start=1):
            for attempt in range(MAX_RETRIES):
                try:
                    records.append(self._parse_card(card))
                    if i % 20 == 0:
                        logger.info(f"  Progress: {i} / {len(cards)} cards parsed …")
                    break
                except StaleElementReferenceException:
                    if attempt == MAX_RETRIES - 1:
                        logger.warning(
                            f"  Skipped card #{i} — stale element after {MAX_RETRIES} retries."
                        )

        logger.info(f"Extraction complete: {len(records)} records collected.")
        return records

---
## § 7 · Execute Scraper

In [7]:
with YouTubeChannelScraper(headless=True) as scraper:
    raw_data = scraper.scrape(max_videos=MAX_VIDEOS)

print(f"\nTotal records collected: {len(raw_data)}")

12:42:44  INFO      ====== WebDriver manager ======
12:42:45  INFO      Get LATEST chromedriver version for google-chrome
12:42:45  INFO      There is no [win64] chromedriver "156.0.8078.47" for browser google-chrome "156.0.8078" in cache
12:42:46  INFO      There is no [win64] chromedriver "156.0.8078.47" for browser google-chrome "156.0.8078" in cache
12:42:46  INFO      Get LATEST chromedriver version for google-chrome
12:42:46  INFO      WebDriver version 156.0.8078.47 selected
12:42:46  INFO      Modern chrome version https://storage.googleapis.com/chrome-for-testing-public/156.0.8078.47/win64/chromedriver-win64.zip
12:42:46  INFO      About to download new driver from https://storage.googleapis.com/chrome-for-testing-public/156.0.8078.47/win64/chromedriver-win64.zip
12:42:47  INFO      Driver downloading response is 200
12:42:50  INFO      Driver has been saved in cache [C:\Users\essam\.wdm\drivers\chromedriver\win64\156.0.8078.47]
12:42:56  INFO      Chrome driver ready  [headle


Total records collected: 1593


---
## § 8 · Build & Clean DataFrame

In [8]:
# ── Build ─────────────────────────────────────────────────────────────────────
df = pd.DataFrame(raw_data)

# ── Type casting ──────────────────────────────────────────────────────────────
df["views"] = pd.to_numeric(df["views"], errors="coerce").astype("Int64")

# ── Drop columns not needed for downstream analysis ───────────────────────────
drop_cols = ["views_raw", "channel", "scraped_at", "thumbnail"]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

# ── Remove cards that failed to load (title == "N/A") ─────────────────────────
before = len(df)
df     = df[df["title"] != "N/A"].reset_index(drop=True)
removed = before - len(df)
if removed:
    logger.info(f"Removed {removed} empty / unloaded cards.")

print(f"Shape   : {df.shape}")
print(f"Columns : {list(df.columns)}")
print()
df.head(10)

Shape   : (1593, 5)
Columns : ['title', 'url', 'views', 'upload_date', 'duration']



,title,url,views,upload_date,duration
0,تدفع أكثر على الخامات ولا الأداء؟ 🤔🔥 | Sony 10...,https://www.youtube.com/watch?v=gZtMcRFioy4,31000,20h ago,"17 minutes, 1 second"
1,بعد قرابة 7 سنوات… هل ما زالت تصمد؟ 🖨️ | Creal...,https://www.youtube.com/watch?v=yTvMM2PNzO4,34000,2d ago,"26 minutes, 14 seconds"
2,ترقية السيرفر بكرت 5090 إضافي! 🚀 | بناء سيرفر ...,https://www.youtube.com/watch?v=C6ibeVePqvw,82000,5d ago,"22 minutes, 56 seconds"
3,أغرب حادثة ذكاء اصطناعي صارت في 2026! | قصص تقنية,https://www.youtube.com/watch?v=J2zK1GBocck,90000,7d ago,"27 minutes, 32 seconds"
4,أحمد أبو زيد في مواجهة أسئلتكم المستفزة! 🔥,https://www.youtube.com/watch?v=ghK9T0SoWhQ&pp...,<NA>,and دروس أونلاين,"1 hour, 7 minutes, 45 seconds"
5,صناديق الأسبوع #238,https://www.youtube.com/watch?v=WqBuX3Jku44,11,N/A,8 minutes
6,بناء جهاز الذكاء الاصطناعي كان أصعب مما توقعت!...,https://www.youtube.com/watch?v=DHsSBVkMrT8,148000,12d ago,"32 minutes, 49 seconds"
7,هل SteamOS هو مستقبل الألعاب على لينكس؟ 🎮 | تج...,https://www.youtube.com/watch?v=CfxQGfNuuOI&pp...,99000,2w ago,"59 minutes, 3 seconds"
8,تقنية خرافية تحت الـ 50 دولار 🔥,https://www.youtube.com/watch?v=KgyJqu3P-cM,109000,2w ago,"13 minutes, 35 seconds"
9,صناديق الأسبوع #237,https://www.youtube.com/watch?v=scHfVCwlGm0,2,N/A,"16 minutes, 1 second"


---
## § 9 · Data Quality Check

In [9]:
print("── Column dtypes ────────────────────────────────────────────────────────────")
print(df.dtypes)
print()
print("── Missing values ───────────────────────────────────────────────────────────")
print(df.isnull().sum())
print()
print("── View count statistics ────────────────────────────────────────────────────")
print(df["views"].describe().apply(lambda x: f"{x:,.0f}"))

── Column dtypes ────────────────────────────────────────────────────────────
title            str
url              str
views          Int64
upload_date      str
duration         str
dtype: object

── Missing values ───────────────────────────────────────────────────────────
title          0
url            0
views          4
upload_date    0
duration       0
dtype: int64

── View count statistics ────────────────────────────────────────────────────
count         1,589
mean        292,393
std       1,095,730
min               1
25%          51,000
50%          90,000
75%         158,000
max      11,000,000
Name: views, dtype: str


---
## § 10 · Quick Exploration

In [10]:
# ── Top 10 most-viewed videos ─────────────────────────────────────────────────
print("── Top 10 Videos by View Count ──────────────────────────────────────────────")
top10 = (
    df[["title", "views", "upload_date", "duration"]]
    .sort_values("views", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
print(top10.to_string())

── Top 10 Videos by View Count ──────────────────────────────────────────────
                 title     views upload_date                duration
0  صناديق الأسبوع #194  11000000         N/A   10 minutes, 6 seconds
1  صناديق الأسبوع #197  11000000         N/A  11 minutes, 24 seconds
2  صناديق الأسبوع #196  11000000         N/A    11 minutes, 1 second
3  صناديق الأسبوع #195  11000000         N/A  17 minutes, 22 seconds
4  صناديق الأسبوع #201  10000000         N/A   16 minutes, 2 seconds
5  صناديق الأسبوع #202  10000000         N/A  37 minutes, 28 seconds
6  صناديق الأسبوع #198  10000000         N/A  31 minutes, 43 seconds
7  صناديق الأسبوع #199  10000000         N/A  19 minutes, 11 seconds
8  صناديق الأسبوع #205   9000000         N/A   20 minutes, 9 seconds
9  صناديق الأسبوع #204   9000000         N/A   9 minutes, 34 seconds


In [11]:
# ── Upload date distribution ──────────────────────────────────────────────────
print("── Upload Date Distribution (top 15 buckets) ────────────────────────────────")
print(df["upload_date"].value_counts().head(15).to_string())

── Upload Date Distribution (top 15 buckets) ────────────────────────────────
upload_date
N/A        140
8y ago     126
9y ago     125
2y ago     120
11y ago    120
3y ago     119
4y ago     115
5y ago     114
1y ago     112
10y ago    109
7y ago     106
6y ago     101
12y ago     39
9mo ago     15
3mo ago     13


---
## § 11 · Export to CSV

`utf-8-sig` encoding ensures Arabic titles render correctly when the file is opened in Microsoft Excel.

In [12]:
df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

file_size_kb = CSV_PATH.stat().st_size / 1024
logger.info(f"Saved {len(df)} rows to {CSV_PATH}  ({file_size_kb:.1f} KB)")

print(f"\n{'─'*55}")
print(f"  File    : {CSV_PATH}")
print(f"  Rows    : {len(df):,}")
print(f"  Columns : {len(df.columns)}")
print(f"  Size    : {file_size_kb:.1f} KB")
print(f"{'─'*55}")

12:47:42  INFO      Saved 1593 rows to output\AndroidBasha_videos_20261021_124244.csv  (237.5 KB)



───────────────────────────────────────────────────────
  File    : output\AndroidBasha_videos_20261021_124244.csv
  Rows    : 1,593
  Columns : 5
  Size    : 237.5 KB
───────────────────────────────────────────────────────


---
## § 12 · Reload & Verify

In [13]:
# Read back from disk to confirm integrity
df_check = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

print(f"Reloaded shape : {df_check.shape}")
print(f"Columns        : {list(df_check.columns)}")
print()
df_check.head(3)

Reloaded shape : (1593, 5)
Columns        : ['title', 'url', 'views', 'upload_date', 'duration']



,title,url,views,upload_date,duration
0,تدفع أكثر على الخامات ولا الأداء؟ 🤔🔥 | Sony 10...,https://www.youtube.com/watch?v=gZtMcRFioy4,31000.0,20h ago,"17 minutes, 1 second"
1,بعد قرابة 7 سنوات… هل ما زالت تصمد؟ 🖨️ | Creal...,https://www.youtube.com/watch?v=yTvMM2PNzO4,34000.0,2d ago,"26 minutes, 14 seconds"
2,ترقية السيرفر بكرت 5090 إضافي! 🚀 | بناء سيرفر ...,https://www.youtube.com/watch?v=C6ibeVePqvw,82000.0,5d ago,"22 minutes, 56 seconds"


---
## Pipeline Summary

| Step | Component | Status |
|------|-----------|--------|
| 1 | Dependency installation | ✅ |
| 2 | Imports & logging | ✅ |
| 3 | Configuration | ✅ |
| 4 | Headless Chrome driver | ✅ |
| 5 | Helper utilities | ✅ |
| 6 | `YouTubeChannelScraper` class | ✅ |
| 7 | Scraper execution | ✅ |
| 8 | DataFrame build & cleaning | ✅ |
| 9 | Data quality check | ✅ |
| 10 | Exploratory analysis | ✅ |
| 11 | CSV export | ✅ |
| 12 | File verification | ✅ |

<div style="
    background:#161b22;
    border:1px solid #21262d;
    border-left:4px solid #00b4d8;
    border-radius:8px;
    padding:16px 20px;
    font-family:'JetBrains Mono', monospace;
    font-size:12px;
    color:#8b949e;
    margin-top:16px;
">
<span style="color:#00b4d8;">Output file:</span>
<code style="color:#e6edf3; margin-left:8px;">output/&lt;channel&gt;_videos_&lt;timestamp&gt;.csv</code><br>
<span style="color:#00b4d8;">Columns:</span>
<code style="color:#e6edf3; margin-left:8px;">title · url · views · upload_date · duration</code>
</div>